# XGBOOST Model - Analysis

In [ ]:
import xgboost as xgb
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

train_df = pd.read_csv('train.csv')
val_df   = pd.read_csv('val.csv')
test_df  = pd.read_csv('test.csv')

target = 'target_is_fraud'

In [ ]:
X_train, y_train = train_df.drop(columns=[target]), train_df[target]
X_val, y_val = val_df.drop(columns=[target]), val_df[target]
X_test, y_test  = test_df.drop(columns=[target]), test_df[target]

In [ ]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

model = xgb.XGBClassifier(
    n_estimators=500,                       # nb max d'arbres construits
    max_depth=5,                            # profondeur max de chaque arbre
    learning_rate=0.05,                     
    subsample=0.8,                          # 80% des lignes du train aléatoirement pour construire arbre
    colsample_bytree=0.8,                   # 80% des features aléatoirement pour construire arbre (réduit l'overfitting)
    scale_pos_weight=scale_pos_weight,      
    random_state=42,                        
    eval_metric='aucpr',                    
    early_stopping_rounds=20               # arrête l'entraînement si la performance sur le val set ne s'améliore pas pendant 20 tours 
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=50                 
)

In [ ]:
y_pred      = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

In [ ]:
print(f"Meilleur nombre d'arbres : {model.best_iteration}")
print(f"Accuracy  : {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC-AUC   : {roc_auc_score(y_test, y_pred_proba):.4f}")
print(classification_report(y_test, y_pred))